# RoPE: position as a rotation

**RoFormer: Enhanced Transformer with Rotary Position Embedding** · Su et al. · 2021

[Original paper](https://arxiv.org/abs/2104.09864)

**Learning goal:** Implement rotary embeddings and test norm preservation and relative-position dot products.

## Before starting

Attention lesson, 2D rotations, and tensor shapes.

Allow roughly 40–60 minutes including exercises. CPU, PyTorch, and Matplotlib; no model or dataset downloads. This is a study-time estimate, not a runtime benchmark.

## Problem and idea

RoPE encodes position by rotating coordinate pairs of Q and K rather than adding a position vector to a token. Different pairs use different frequencies, representing offsets at multiple scales. In the usual application, V is not rotated.

Distinguish the mathematical relative-position property from a trained model’s ability to handle longer contexts. The former can be tested directly; the latter needs evaluation. Frequency changes and context scaling do not guarantee quality.

## Reproduction boundary

Rotation and its properties only; no RoFormer LM or context-extension method.

The code is an original educational implementation of mathematical ideas from the paper, not a copy of an author repository. Passing assertions verifies the stated implementation properties, not the paper’s results.

## Formula → tensors

For pair `(a,b)` and angle `phi = position * theta_i`, rotate to `(a*cos(phi)-b*sin(phi), a*sin(phi)+b*cos(phi))`, with `theta_i = base^(-2i/D)`.

`R(m)q · R(n)k = q · R(n-m)k`, so a common shift preserves the dot product. D must be even. We use adjacent coordinate pairs; other layouts need consistent coordinate and weight permutations.

In [ ]:
%matplotlib inline
import math
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| CPU | seed=42')

## Step 1

Implement rotations in float64 for a precise numerical check.

In [ ]:
def rope(x, positions, base=10000.):
    width = x.shape[-1]
    assert width % 2 == 0
    frequency = base ** (-torch.arange(0, width, 2, dtype=x.dtype) / width)
    angle = positions.to(x.dtype)[..., None] * frequency
    even, odd = x[..., 0::2], x[..., 1::2]
    result = torch.stack([even * angle.cos() - odd * angle.sin(),
                          even * angle.sin() + odd * angle.cos()], dim=-1)
    return result.flatten(-2)
x = torch.randn(8, dtype=torch.float64)
rotated = rope(x, torch.tensor(7))
print('norm error:', (x.norm() - rotated.norm()).abs().item())
assert torch.allclose(x.norm(), rotated.norm(), atol=1e-12)

## Step 2

Test the relative-position identity and a common position shift.

In [ ]:
q = torch.randn(8, dtype=torch.float64); k = torch.randn_like(q)
m, n, offset = 5, 11, 20
left = rope(q, torch.tensor(m)) @ rope(k, torch.tensor(n))
right = q @ rope(k, torch.tensor(n - m))
shifted = rope(q, torch.tensor(m + offset)) @ rope(k, torch.tensor(n + offset))
print('relative error:', abs(left - right).item(), 'common-shift error:', abs(left - shifted).item())
assert torch.allclose(left, right, atol=1e-12)
assert torch.allclose(left, shifted, atol=1e-12)

## Step 3

Plot the offset dependence and check differentiability.

In [ ]:
positions = torch.arange(128)
scores = torch.stack([rope(q, torch.tensor(0)) @ rope(k, p) for p in positions])
plt.plot(positions, scores, color='#d73535'); plt.xlabel('relative offset'); plt.ylabel('Q dot K'); plt.show()
differentiable = torch.randn(2, 6, 8, requires_grad=True)
result = rope(differentiable, torch.arange(6))
result.mul(torch.randn_like(result)).sum().backward()
assert torch.isfinite(differentiable.grad).all()
print('gradient norm:', differentiable.grad.norm().item())

## Reading the result

A single Q/K pair can produce oscillations, not monotonic decay. Norm preservation checks the rotation; common-shift invariance checks positional structure. Both matter: preserving norms alone does not establish correct RoPE.

## Independent experiments

1. Rotate Q only: does common-shift invariance survive?
2. Change base from 10000 to 100.
3. Use D=7 and explain the error.

Write a prediction first, then change one factor at a time. Reset the kernel and rerun all cells for a controlled comparison. Save a copy for your changes.

## Reasoning check — after experimenting

Without consistently rotating K, dot products depend on Q’s absolute position. Base changes the frequency grid. Odd width cannot form complete pairs; partial rotary requires an explicitly selected even-sized subspace.

## What to record in your lab notes

- Hypothesis and changed parameter.
- Seed, data size, step budget, and precision.
- Before/after diagnostics, plot, and explanation of unexpected results.
- What this experiment **does not** establish.

Use “Open Studio workspace” above the lesson to continue in the related tools.